In [1]:
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dropout, Flatten, Dense
from keras.callbacks import ReduceLROnPlateau, Callback, ModelCheckpoint
from sklearn.metrics import cohen_kappa_score
from tensorflow.keras.utils import to_categorical
from keras import backend as K
from keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import os
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from keras.preprocessing.image import ImageDataGenerator
from keras.models import load_model
from tensorflow.keras.applications import DenseNet121
from tqdm import tqdm
from sklearn.utils import class_weight


import keras
from keras import Input
from keras.models import Model
from keras.utils import *
from keras.layers import *


import matplotlib.pyplot as plt

import gc

2026-04-28 02:50:58.854346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 02:50:58.865290: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777344658.878678      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777344658.882701      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 02:50:58.896655: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ImportError: cannot import name 'ImageDataGenerator' from 'keras.preprocessing.image' (/usr/local/lib/python3.11/dist-packages/keras/api/preprocessing/image/__init__.py)

In [2]:
'''
    Config
'''
IMG_SIZE = 224
BATCH_SIZE = 16

def crop_image_from_gray(img,tol=7):
    if img.ndim ==2:
        mask = img>tol
        return img[np.ix_(mask.any(1),mask.any(0))]
    elif img.ndim==3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img>tol
        
        check_shape = img[:,:,0][np.ix_(mask.any(1),mask.any(0))].shape[0]
        if (check_shape == 0): 
            return img 
        else:
            img1=img[:,:,0][np.ix_(mask.any(1),mask.any(0))]
            img2=img[:,:,1][np.ix_(mask.any(1),mask.any(0))]
            img3=img[:,:,2][np.ix_(mask.any(1),mask.any(0))]
            img = np.stack([img1,img2,img3],axis=-1)
        return img
    
def load_ben_color(image, sigmaX=10):
    image = crop_image_from_gray(image)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = cv2.addWeighted ( image,4, cv2.GaussianBlur( image , (0,0) , sigmaX) ,-4 ,128)
        
    return image.astype('float32')

'''
    Preprocessing for ImageDataGenerator since ImageDataGenerator reads images in rgb mode, while opencv in bgr
'''
def preprocessing(image, sigmaX=10):
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    image = crop_image_from_gray(image).astype('uint8')
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image=cv2.addWeighted ( image,4, cv2.GaussianBlur( image , (0,0) , sigmaX) ,-4 ,128)
        
    return image.astype('float32') / 255.

## load train and test image 

In [3]:
model = keras.models.load_model('../input/eff-b0-model/eff_b0_model')

NameError: name 'keras' is not defined

In [4]:
test_csv = pd.read_csv('../input/aptos2019-blindness-detection/sample_submission.csv')
id_code = test_csv['id_code']
test_prediction = np.empty(len(id_code), dtype='float32')
for i in range(len(id_code)):
    img = cv2.imread('../input/aptos2019-blindness-detection/test_images/{}.png'.format(id_code[i]))
    img = load_ben_color(img)
    X = np.array([img])
    pred = model.predict(X)
    test_prediction[i] = np.argmax(pred).astype('int64')
test_prediction = test_prediction.astype('int64')

NameError: name 'model' is not defined

In [5]:
# prediction = predict(test_prediction).astype('uint8')
test_csv['diagnosis'] = test_prediction
test_csv.to_csv("submission.csv", index=False)
unique, counts = np.unique(test_prediction, return_counts=True)
tmp = dict(zip(unique, counts))
print(tmp)
print('Done!')

{-9.3326715e+33: 2, -9.3319485e+33: 4, 0.0: 264, 1e-45: 7, 3e-45: 5, 4e-45: 29, 6e-45: 1, 2.1e-44: 3, 2.5e-44: 5, 2.8e-44: 1, 3.6e-44: 2, 3.8e-44: 4, 4.2e-44: 2, 4.3e-44: 1, 5.3e-44: 1, 5.5e-44: 2, 6.6e-44: 1, 7.8e-44: 1, 8e-44: 2, 1.16e-43: 1, 1.26e-43: 1, 1.4e-43: 2, 1.42e-43: 3, 2.24e-43: 1, 2.33e-43: 2, 2.4e-43: 2, 2.41e-43: 1, 3.15e-43: 1, 1.235e-42: 1, 4.5916e-41: 7, 4.052211e-33: 2, 0.010972723: 2, nan: 4}
Done!
